<a href="https://colab.research.google.com/github/KrishViradiya/group8_hadoop_mapReduce/blob/main/promting_technique/keval%20kathirya/Copy_of_%5BREAD_ONLY%5DDS4SE26Week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [ ]:
!pip install -U tokenizers transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [ ]:
# A smaller model that fits on a free Colab GPU
model_name = "ibm-granite/granite-3.3-8b-instruct"

# 2. Hardware optimization (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
pip install -U bitsandbytes>=0.46.1

# 3. Load the tokenizer & model

In [ ]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

In [ ]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16
)

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
        "role": "system",
        "content": (
            "You are an expert software architect specializing in the Apache Hadoop ecosystem. "
            "Your task is to analyze Java source code from the Hadoop MapReduce 'Client core' component "
            "and identify its architectural purpose."
        )
    },
    {
        "role": "user",
        "content": """Please analyze the following classes from a single cluster identified during architectural recovery.

        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
       "role": "system",
        "content": (
            "You are a Senior Security Architect specializing in distributed systems and Kerberos authentication. "
            "Your task is to analyze Java source code from the Hadoop MapReduce 'Client core' component "
            "and identify how this cluster manages secure job submission."
        )
    },
    {
        "role": "user",
        "content": """Analyze the following classes for security-critical functions:
        <source_code>

        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
       "role": "system",
        "content": (
            "You are a Principal Software Engineer focused on design patterns and system maintainability. "
            "Your task is to analyze the provided Hadoop MapReduce classes and identify structural "
            "patterns (like Factory, Proxy, or Bridge) that facilitate job context management."
        )
    },
    {
        "role": "user",
        "content": """Identify design patterns used in the following classes:
        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())